In [2]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.cluster.hierarchy import linkage
from scipy.spatial.distance import pdist


In [3]:
calls = pd.read_pickle("/ceph/MethDev/pbio/kay/data/calls_loose.pkl")
calls

,chr,start,end,X2,df,delta_max,hi_cluster,lo_cluster,delta_max_trim,top_cluster,...,df_loco,phi,pval,p_loco,qval,neighbor_support,dominance_blocked,reason,category,call_reason
85,1,77801,77900,283.205748,16,0.509852,16,0,0.258787,16,...,15.0,1.536786,0.000000e+00,1.479381e-03,0.000000e+00,0,False,no_neighbor_support,euc_gene,rescue_isolated
125,1,117201,117300,209.579299,16,0.469321,16,0,0.315957,16,...,15.0,1.536786,0.000000e+00,9.155010e-12,0.000000e+00,1,False,ok,euc_gene,main
129,1,121701,121800,121.351245,16,0.345870,4,16,0.321142,9,...,15.0,1.536786,2.556039e-10,7.882232e-08,1.349449e-08,1,False,ok,euc_gene,main
142,1,123101,123200,60.386475,16,0.280947,0,9,0.235865,9,...,15.0,1.536786,9.862091e-04,2.243899e-02,1.093957e-02,1,False,ok,euc_gene,main
294,1,243401,243500,69.454337,16,0.201690,4,9,0.186608,0,...,15.0,1.536786,1.295724e-04,1.617305e-02,1.967793e-03,2,False,weak_effect,euc_gene,main
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
304611,5,15242801,15242900,77.878127,16,0.302944,13,12,0.154973,12,...,15.0,1.544736,1.968215e-05,3.571681e-02,1.171208e-03,2,False,weak_effect,het_te,main
304613,5,15243001,15243100,86.855885,16,0.244768,8,9,0.176046,12,...,15.0,1.544736,2.232159e-06,2.235987e-03,1.974125e-04,2,False,weak_effect,het_te,main
304627,5,15259701,15259800,70.739222,16,0.413612,8,6,0.340276,6,...,15.0,1.544736,1.047749e-04,9.935832e-04,4.410364e-03,1,False,ok,het_te,main
304628,5,15259801,15259900,67.105317,16,0.465839,5,4,0.305060,4,...,15.0,1.544736,2.397276e-04,1.577884e-02,8.303919e-03,2,False,ok,het_te,main


In [4]:
df = pd.read_csv('/ceph/MethDev/pbio/kay/data/annotated_filtered_col.CG_2.fast.tsv', sep='\t')      # or read_parquet / feather …
df

,cluster,chr,start,end,score,c,t,n,flag_euc_gene,flag_het_gene,flag_euc_TE,flag_het_TE,score_masked
0,0,1,101,200,0.8951,350,41,6,False,False,False,False,0.8951
1,0,1,301,400,0.5487,62,51,2,False,False,False,False,0.5487
2,0,1,401,500,0.8246,47,10,1,False,False,False,False,0.8246
3,0,1,501,600,0.7206,98,38,3,False,False,False,False,0.7206
4,0,1,601,700,0.8982,203,23,6,False,False,False,False,0.8982
...,...,...,...,...,...,...,...,...,...,...,...,...,...
5925984,16,5,26974801,26974900,0.8462,22,4,7,False,False,False,False,0.8462
5925985,16,5,26974901,26975000,0.7500,3,1,2,False,False,False,False,NaN
5925986,16,5,26975101,26975200,1.0000,4,0,4,False,False,False,False,NaN
5925987,16,5,26975201,26975300,1.0000,8,0,8,False,False,False,False,1.0000


In [6]:
win_list = calls[["chr", "start", "end"]]
win_list

,chr,start,end
85,1,77801,77900
125,1,117201,117300
129,1,121701,121800
142,1,123101,123200
294,1,243401,243500
...,...,...,...
304611,5,15242801,15242900
304613,5,15243001,15243100
304627,5,15259701,15259800
304628,5,15259801,15259900


In [26]:
df_sub = (df
          .merge(win_list[['chr','start','end']], on=['chr','start','end'], how='inner')
          .copy())
df_sub

,cluster,chr,start,end,score,c,t,n,flag_euc_gene,flag_het_gene,flag_euc_TE,flag_het_TE,score_masked
0,0,1,76901,77000,0.9121,83,8,2,False,False,True,False,0.9121
1,1,1,76901,77000,0.8169,58,13,2,False,False,True,False,0.8169
2,2,1,76901,77000,0.9750,39,1,2,False,False,True,False,0.9750
3,3,1,76901,77000,0.8000,20,5,2,False,False,True,False,0.8000
4,4,1,76901,77000,0.5200,13,12,2,False,False,True,False,0.5200
...,...,...,...,...,...,...,...,...,...,...,...,...,...
137411,12,5,26961801,26961900,0.6667,6,3,5,True,False,False,False,0.6667
137412,13,5,26961801,26961900,1.0000,35,0,6,True,False,False,False,1.0000
137413,14,5,26961801,26961900,1.0000,11,0,6,True,False,False,False,1.0000
137414,15,5,26961801,26961900,1.0000,11,0,5,True,False,False,False,1.0000


In [27]:
df_sub.iloc[:30]

,cluster,chr,start,end,score,c,t,n,flag_euc_gene,flag_het_gene,flag_euc_TE,flag_het_TE,score_masked
0,0,1,76901,77000,0.9121,83,8,2,False,False,True,False,0.9121
1,1,1,76901,77000,0.8169,58,13,2,False,False,True,False,0.8169
2,2,1,76901,77000,0.9750,39,1,2,False,False,True,False,0.9750
3,3,1,76901,77000,0.8000,20,5,2,False,False,True,False,0.8000
4,4,1,76901,77000,0.5200,13,12,2,False,False,True,False,0.5200
5,5,1,76901,77000,1.0000,16,0,2,False,False,True,False,1.0000
6,6,1,76901,77000,0.5263,10,9,2,False,False,True,False,0.5263
7,7,1,76901,77000,0.8519,23,4,2,False,False,True,False,0.8519
8,8,1,76901,77000,0.6000,3,2,2,False,False,True,False,0.6000
9,9,1,76901,77000,0.6250,5,3,2,False,False,True,False,0.6250


In [28]:
val_col = 'score_masked'

In [29]:
# Pivot to matrix
mat = (df_sub
       .pivot_table(index=['chr','start','end'],
                    columns='cluster',
                    values=val_col,
                    aggfunc='first')
       .sort_index())

# Attach row metadata for colors
row_meta = (win_list.set_index(['chr','start','end'])
                     .reindex(mat.index))

In [30]:
mat

cluster                    0       1       2       3       4       5       6   \
chr start    end                                                                
1   76901    77000     0.9121  0.8169  0.9750  0.8000  0.5200  1.0000  0.5263   
    77001    77100     0.8858  0.8272  0.9014  0.9194  0.2636  0.8718  0.5462   
    77201    77300     0.9338  0.9451  0.8760  0.9850  0.3282  0.9259  0.8276   
    77801    77900     0.0028  0.0034  0.0072  0.0000  0.0000  0.0000  0.0000   
    117201   117300    0.0016  0.0059  0.0000  0.0000  0.0043  0.0000  0.0194   
...                       ...     ...     ...     ...     ...     ...     ...   
5   26886301 26886400  0.5933  0.5308  0.4818  0.5657  0.1942  0.4535  0.2333   
    26886801 26886900  0.5109  0.5105  0.4754  0.5217  0.2880  0.5077  0.2727   
    26893101 26893200  0.7799  0.6806  0.9275  0.8913  0.8254  1.0000  0.8041   
    26914001 26914100  0.5648  0.4819  0.4608  0.3719  0.5020  0.6180  0.4161   
    26961801 26961900  0.9357  0.9227  0.9270  0.9348  0.8661  0.8730  0.8800   

cluster                    7       8       9       10      11      12      13  \
chr start    end                                                                
1   76901    77000     0.8519  0.6000  0.6250     NaN  0.8571  0.8000     NaN   
    77001    77100     0.8105  0.4000  0.8649  0.6957  0.8974  0.7391  0.3667   
    77201    77300     0.8480  0.5741  1.0000  0.5857  1.0000  0.5625  0.3810   
    77801    77900     0.0000  0.0000  0.0000  0.0385  0.1000  0.0000  0.0000   
    117201   117300    0.0000  0.0000  0.0571  0.0000  0.0000  0.0000  0.0000   
...                       ...     ...     ...     ...     ...     ...     ...   
5   26886301 26886400  0.4235  0.4127  0.4500  0.5200  0.6818  0.6250  0.0938   
    26886801 26886900  0.5385  0.3333  0.6250  0.1923  0.5455  0.5667  0.2593   
    26893101 26893200  0.7407  0.9000     NaN  0.2500  0.8372  0.9565  0.8421   
    26914001 26914100  0.5461  0.5625  0.2143  0.5526  0.4068  0.4286  0.4167   
    26961801 26961900  0.9274  0.8605  0.5714  0.8750  0.5682  0.6667  1.0000   

cluster                    14      15      16  
chr start    end                               
1   76901    77000        NaN     NaN     NaN  
    77001    77100     0.9130  0.8750  0.8000  
    77201    77300     0.7241  0.9583  0.8846  
    77801    77900     0.0000  0.0000  0.4286  
    117201   117300    0.0000  0.2000  0.2500  
...                       ...     ...     ...  
5   26886301 26886400  0.2857  0.9000  0.6667  
    26886801 26886900     NaN  0.4615     NaN  
    26893101 26893200  1.0000  1.0000     NaN  
    26914001 26914100  0.3182  0.4667  0.1613  
    26961801 26961900  1.0000  1.0000  1.0000  

[8116 rows x 17 columns]

In [31]:
# Center per row (subtract row median so 0 = typical for that window)
row_med = np.nanmedian(mat.values, axis=1, keepdims=True)
mat_centered = mat.values - row_med


In [32]:
row_std = np.nanstd(mat.values, axis=1, keepdims=True)
mat_centered = mat_centered / np.where(row_std==0, 1, row_std)
mat_centered = pd.DataFrame(mat_centered, index=mat.index, columns=mat.columns)
mat_centered

cluster                      0         1         2         3         4   \
chr start    end                                                          
1   76901    77000     0.650078  0.052997  1.044578 -0.052997 -1.809118   
    77001    77100     0.284154  0.000000  0.359799  0.447082 -2.732921   
    77201    77300     0.272759  0.326084  0.000000  0.514373 -2.585077   
    77801    77900     0.027563  0.033470  0.070877  0.000000  0.000000   
    117201   117300    0.022095  0.081475  0.000000  0.000000  0.059380   
...                         ...       ...       ...       ...       ...   
5   26886301 26886400  0.576461  0.253333  0.000000  0.433767 -1.486907   
    26886801 26886900  0.024759  0.021664 -0.249914  0.108322 -1.699881   
    26893101 26893200 -0.343810 -0.892690  0.472048  0.271953 -0.092309   
    26914001 26914100  0.869201  0.176347  0.000000 -0.743000  0.344337   
    26961801 26961900  0.096702  0.000000  0.031986  0.090007 -0.421025   

cluster                      5         6         7         8         9   \
chr start    end                                                          
1   76901    77000     1.201375 -1.769606  0.272512 -1.307369 -1.150573   
    77001    77100     0.216267 -1.362581 -0.080979 -2.071512  0.182809   
    77201    77300     0.235479 -0.228400 -0.132132 -1.424671  0.585158   
    77801    77900     0.000000  0.000000  0.000000  0.000000  0.000000   
    117201   117300    0.000000  0.267899  0.000000  0.000000  0.788508   
...                         ...       ...       ...       ...       ...   
5   26886301 26886400 -0.146312 -1.284758 -0.301414 -0.357251 -0.164408   
    26886801 26886900  0.000000 -1.818261  0.238308 -1.349382  0.907583   
    26893101 26893200  0.872792 -0.210045 -0.560488  0.320042       NaN   
    26914001 26914100  1.313830 -0.373589  0.712912  0.849978 -2.060173   
    26961801 26961900 -0.369698 -0.317628  0.034961 -0.462681 -2.613180   

cluster                      10        11        12        13        14  \
chr start    end                                                          
1   76901    77000          NaN  0.305126 -0.052997       NaN       NaN   
    77001    77100    -0.637649  0.340403 -0.427201 -2.232985  0.416048   
    77201    77300    -1.369930  0.585158 -1.479412 -2.335913 -0.716819   
    77801    77900     0.378996  0.984405  0.000000  0.000000  0.000000   
    117201   117300    0.000000  0.000000  0.000000  0.000000  0.000000   
...                         ...       ...       ...       ...       ...   
5   26886301 26886400  0.197496  1.034010  0.740351 -2.005980 -1.013847   
    26886801 26886900 -2.440338  0.292469  0.456500 -1.921941       NaN   
    26893101 26893200 -3.272830 -0.027085  0.632345  0.000000  0.872792   
    26914001 26914100  0.767237 -0.451316 -0.269118 -0.368575 -1.191808   
    26961801 26961900 -0.354821 -2.636984 -1.904282  0.575004  0.575004   

cluster                      15        16  
chr start    end                           
1   76901    77000          NaN       NaN  
    77001    77100     0.231784 -0.131894  
    77201    77300     0.388375  0.040584  
    77801    77900     0.000000  4.219158  
    117201   117300    2.761848  3.452310  
...                         ...       ...  
5   26886301 26886400  2.162116  0.955943  
    26886801 26886900 -0.357462       NaN  
    26893101 26893200  0.872792       NaN  
    26914001 26914100  0.049310 -2.503131  
    26961801 26961900  0.575004  0.575004  

[8116 rows x 17 columns]

In [36]:
mat_centered.fillna(0.0)

cluster                      0         1         2         3         4   \
chr start    end                                                          
1   76901    77000     0.650078  0.052997  1.044578 -0.052997 -1.809118   
    77001    77100     0.284154  0.000000  0.359799  0.447082 -2.732921   
    77201    77300     0.272759  0.326084  0.000000  0.514373 -2.585077   
    77801    77900     0.027563  0.033470  0.070877  0.000000  0.000000   
    117201   117300    0.022095  0.081475  0.000000  0.000000  0.059380   
...                         ...       ...       ...       ...       ...   
5   26886301 26886400  0.576461  0.253333  0.000000  0.433767 -1.486907   
    26886801 26886900  0.024759  0.021664 -0.249914  0.108322 -1.699881   
    26893101 26893200 -0.343810 -0.892690  0.472048  0.271953 -0.092309   
    26914001 26914100  0.869201  0.176347  0.000000 -0.743000  0.344337   
    26961801 26961900  0.096702  0.000000  0.031986  0.090007 -0.421025   

cluster                      5         6         7         8         9   \
chr start    end                                                          
1   76901    77000     1.201375 -1.769606  0.272512 -1.307369 -1.150573   
    77001    77100     0.216267 -1.362581 -0.080979 -2.071512  0.182809   
    77201    77300     0.235479 -0.228400 -0.132132 -1.424671  0.585158   
    77801    77900     0.000000  0.000000  0.000000  0.000000  0.000000   
    117201   117300    0.000000  0.267899  0.000000  0.000000  0.788508   
...                         ...       ...       ...       ...       ...   
5   26886301 26886400 -0.146312 -1.284758 -0.301414 -0.357251 -0.164408   
    26886801 26886900  0.000000 -1.818261  0.238308 -1.349382  0.907583   
    26893101 26893200  0.872792 -0.210045 -0.560488  0.320042  0.000000   
    26914001 26914100  1.313830 -0.373589  0.712912  0.849978 -2.060173   
    26961801 26961900 -0.369698 -0.317628  0.034961 -0.462681 -2.613180   

cluster                      10        11        12        13        14  \
chr start    end                                                          
1   76901    77000     0.000000  0.305126 -0.052997  0.000000  0.000000   
    77001    77100    -0.637649  0.340403 -0.427201 -2.232985  0.416048   
    77201    77300    -1.369930  0.585158 -1.479412 -2.335913 -0.716819   
    77801    77900     0.378996  0.984405  0.000000  0.000000  0.000000   
    117201   117300    0.000000  0.000000  0.000000  0.000000  0.000000   
...                         ...       ...       ...       ...       ...   
5   26886301 26886400  0.197496  1.034010  0.740351 -2.005980 -1.013847   
    26886801 26886900 -2.440338  0.292469  0.456500 -1.921941  0.000000   
    26893101 26893200 -3.272830 -0.027085  0.632345  0.000000  0.872792   
    26914001 26914100  0.767237 -0.451316 -0.269118 -0.368575 -1.191808   
    26961801 26961900 -0.354821 -2.636984 -1.904282  0.575004  0.575004   

cluster                      15        16  
chr start    end                           
1   76901    77000     0.000000  0.000000  
    77001    77100     0.231784 -0.131894  
    77201    77300     0.388375  0.040584  
    77801    77900     0.000000  4.219158  
    117201   117300    2.761848  3.452310  
...                         ...       ...  
5   26886301 26886400  2.162116  0.955943  
    26886801 26886900 -0.357462  0.000000  
    26893101 26893200  0.872792  0.000000  
    26914001 26914100  0.049310 -2.503131  
    26961801 26961900  0.575004  0.575004  

[8116 rows x 17 columns]

In [1]:

# mat: windows × clusters methylation matrix with NaNs allowed
# (build pivot as before into `mat`, then optional row-centering)
row_med = np.nanmedian(mat.values, axis=1, keepdims=True)
mat_centered = pd.DataFrame(mat.values - row_med, index=mat.index, columns=mat.columns)

# ---- 1) Build a distance-friendly copy (no NaNs) for clustering only ----
# Use z-scored rows (≈ correlation) and fill NaNs with 0 AFTER z-scoring
row_mean = np.nanmean(mat_centered.values, axis=1, keepdims=True)
row_std  = np.nanstd(mat_centered.values, axis=1, keepdims=True)
mat_z = (mat_centered.values - row_mean) / np.where(row_std==0, 1, row_std)
mat_z = np.nan_to_num(mat_z, nan=0.0)   # fill remaining NaNs with 0 for distances

# Compute linkages (Euclidean on z-scores works well and avoids NaN distances)
row_link = linkage(pdist(mat_z, metric='euclidean'), method='average')
col_link = linkage(pdist(mat_z.T, metric='euclidean'), method='average')

# ---- 2) Make a cmap that greys out NaNs in the plot ----
cmap = sns.color_palette("vlag", as_cmap=True)
try:
    cmap = cmap.with_extremes(bad='yellow')  # newer Matplotlib
except Exception:
    cmap.set_bad('yellow')                   # older Matplotlib

# ---- 3) Plot using the original matrix WITH NaNs, but our precomputed linkages ----
g = sns.clustermap(
    mat_centered,               # keeps NaNs for display
    row_linkage=row_link,
    col_linkage=col_link,
    cmap=cmap, center=0,
    xticklabels=True, yticklabels=False,
    figsize=(12, 10)
)
g.ax_heatmap.set_xlabel('cluster')
g.ax_heatmap.set_ylabel('windows (centered)')
plt.show()


NameError: name 'np' is not defined

In [45]:
g.fig.savefig("/ceph/MethDev/pbio/kay/data/figures/clustermap_avg.png", dpi=300, bbox_inches="tight")

In [46]:
from scipy.cluster.hierarchy import fcluster

k_rows = 6    # choose how many window clusters you want
k_cols = 4    # (optional) how many groups of cell clusters

row_labels = fcluster(row_link, t=k_rows, criterion='maxclust')   # length = n_rows
col_labels = fcluster(col_link, t=k_cols, criterion='maxclust')   # length = n_cols

# Attach labels to convenient tables
rows_annot = (mat_centered
              .assign(row_cluster=row_labels)
              .rename_axis(['chr','start','end']))        # keep genomic coords

cols_annot = pd.DataFrame({'cluster': mat_centered.columns,
                           'col_cluster': col_labels}).set_index('cluster')


In [48]:
win_idx

MultiIndex([(1,   281901,   282000),
            (1,  1199701,  1199800),
            (1,  1809501,  1809600),
            (1,  2049701,  2049800),
            (1,  5867201,  5867300),
            (1,  5912201,  5912300),
            (1,  6279801,  6279900),
            (1,  6372601,  6372700),
            (1,  6551201,  6551300),
            (1,  8335201,  8335300),
            ...
            (5, 15865101, 15865200),
            (5, 16104601, 16104700),
            (5, 16604801, 16604900),
            (5, 18846501, 18846600),
            (5, 19151401, 19151500),
            (5, 20219001, 20219100),
            (5, 20564001, 20564100),
            (5, 20995101, 20995200),
            (5, 22262901, 22263000),
            (5, 25641901, 25642000)],
           names=['chr', 'start', 'end'], length=352)

In [47]:
# Pick, say, row-cluster 3
cid = 3
win_idx = rows_annot.index[rows_annot['row_cluster'] == cid]  # list of (chr,start,end)
sub = mat_centered.loc[win_idx]                                # windows in this group

print(f"Row cluster {cid}: {len(sub)} windows")
print("A few members:")
print(pd.DataFrame(win_idx, columns=['chr','start','end']).head())

# Aggregate profile across cell clusters (mean pattern for this window-cluster)
profile = np.nanmean(sub.values, axis=0)                       # length = n_clusters
profile = pd.Series(profile, index=mat_centered.columns)
profile.head()


Row cluster 3: 352 windows
A few members:


ValueError: Shape of passed values is (352, 1), indices imply (352, 3)